# TABELA ECONOMIA


In [ ]:
!pip install mysql-connector-python
import mysql.connector
import pandas as pd

# --- CONFIGURAÇÃO ---

MESES_MANDATO = 44

def conectar_banco():
    return mysql.connector.connect(
        host="54.198.148.230",
        port=3306,
        user="root",
        password="lumina1234",
        database="Lumina2",
        connect_timeout=30
    )

def main():
    conn = conectar_banco()
    cursor = conn.cursor(dictionary=True)

    print("Configurando estrutura do banco...")
    comandos_sql = [

        "ALTER TABLE estado ADD COLUMN IF NOT EXISTS cota DECIMAL(10, 2);",
        """
        CREATE TABLE IF NOT EXISTS economia (
            cd_economia INT AUTO_INCREMENT PRIMARY KEY,
            despesa_total DECIMAL(10, 2),
            economia DECIMAL(10, 2),
            fk_deputado INT,
            fk_estado INT,
            FOREIGN KEY (fk_deputado) REFERENCES deputado(cd_deputado),
            FOREIGN KEY (fk_estado) REFERENCES estado(cd_estado)
        );
        """
    ]

    for cmd in comandos_sql:
        try:
            cursor.execute(cmd)
        except mysql.connector.Error as err:

            pass

    conn.commit()


    cotas_estados = {
        'AC': 57359.87, 'AL': 53164.36, 'AM': 56151.46, 'AP': 55929.26,
        'BA': 50965.29, 'CE': 54879.34, 'DF': 41612.55, 'ES': 49160.15,
        'GO': 46979.73, 'MA': 54537.99, 'MG': 47645.91, 'MS': 52707.93,
        'MT': 51439.83, 'PA': 54624.17, 'PB': 54402.48, 'PE': 53997.81,
        'PI': 53195.84, 'PR': 50807.19, 'RJ': 47267.41, 'RN': 55198.09,
        'RO': 56267.90, 'RR': 58474.70, 'RS': 53086.78, 'SC': 51951.42,
        'SP': 48727.46, 'SE': 52248.86, 'TO': 51525.80
    }

    print("Atualizando cotas mensais na tabela 'estado'...")
    for uf, cota in cotas_estados.items():
        cursor.execute("UPDATE estado SET cota = %s WHERE uf = %s", (cota, uf))
    conn.commit()


    print(f"Lendo CSV e calculando economia para o período de {MESES_MANDATO} meses...")
    try:
        df_despesas = pd.read_csv('modulo3_despesas_completa.csv')
    except FileNotFoundError:
        print("Erro: Arquivo 'modulo3_despesas_completa.csv' não encontrado.")
        return


    cursor.execute("""
        SELECT d.cd_deputado, d.fk_estado, e.cota
        FROM deputado d
        JOIN estado e ON d.fk_estado = e.cd_estado
    """)
    deputados_db = cursor.fetchall()

    mapa_deputados = {
        row['cd_deputado']: {'fk_estado': row['fk_estado'], 'cota_mensal': float(row['cota'])}
        for row in deputados_db
    }


    dados_insercao = []

    for index, row in df_despesas.iterrows():
        id_deputado = row['deputado_id']
        despesa_total_acumulada = float(row['Total_Gasto_Historico'])

        if id_deputado in mapa_deputados:
            fk_estado = mapa_deputados[id_deputado]['fk_estado']
            cota_mensal = mapa_deputados[id_deputado]['cota_mensal']

            # CÁLCULO CORRIGIDO:

            cota_total_periodo = cota_mensal * MESES_MANDATO


            economia_real = cota_total_periodo - despesa_total_acumulada

            dados_insercao.append((despesa_total_acumulada, economia_real, id_deputado, fk_estado))
        else:
            print(f"Aviso: Deputado ID {id_deputado} não encontrado no banco. Pulando...")

    if dados_insercao:

        cursor.execute("DELETE FROM economia")

        query_insert = """
            INSERT INTO economia (despesa_total, economia, fk_deputado, fk_estado)
            VALUES (%s, %s, %s, %s)
        """
        cursor.executemany(query_insert, dados_insercao)
        conn.commit()
        print(f"Sucesso! {len(dados_insercao)} registros de economia inseridos.")

    cursor.close()
    conn.close()
    print("Processo finalizado.")

if __name__ == "__main__":
    main()

Configurando estrutura do banco...
Atualizando cotas mensais na tabela 'estado'...
Lendo CSV e calculando economia para o período de 44 meses...
Erro: Arquivo 'modulo3_despesas_completa.csv' não encontrado.


PESO PARA TEMAS

In [ ]:
pesos_temas_completos = {
    '39': 1.0, '44': 1.0, '46': 1.0, '52': 1.0, '56': 1.0, '57': 1.0, '58': 1.0, '61': 1.0, '64': 1.0, '67': 1.0,
    '34': 0.5, '35': 0.5, '37': 0.5, '40': 0.5, '41': 0.5, '42': 0.5, '43': 0.5, '48': 0.5, '51': 0.5, '53': 0.5,
    '54': 0.5, '55': 0.5, '60': 0.5, '62': 0.5, '66': 0.5, '68': 0.5, '70': 0.5, '74': 0.5, '76': 0.5,
    '72': 0.1, '85': 0.1, '86': 0.1
}

# Transformar dicionário para lista de tuplas.
dados_update = [(peso)]

#Desempenho


In [ ]:

import mysql.connector
import pandas as pd
import requests
import time

# =============================================================================
# Conexão com o banco de dados
# =============================================================================
print("=" * 60)
print("ETAPA 0 — Conectando ao banco de dados...")
print("=" * 60)

config = {
    "host": "54.198.148.230",
    "port": 3306,
    "user": "root",
    "password": "lumina1234",
    "database": "Lumina2"
}

conn = None

try:
    conn = mysql.connector.connect(**config)
    cursor = conn.cursor()
    print("✓ Conexão estabelecida com sucesso.\n")

    # =========================================================================
    # anos e IDs válidos do banco
    # =========================================================================
    print("=" * 60)
    print("ETAPA 1 — Carregando anos válidos da tabela proposicoes...")
    print("=" * 60)

    cursor.execute("""
        SELECT DISTINCT ano
        FROM proposicoes
        WHERE ano IS NOT NULL AND ano > 1900
        ORDER BY ano
    """)
    anos_no_banco = [row[0] for row in cursor.fetchall()]

    if not anos_no_banco:
        print("✗ Nenhum ano válido encontrado. Encerrando.")
    else:
        print(f"✓ Anos encontrados: {anos_no_banco}\n")

        for ano in anos_no_banco:
            print("\n" + "=" * 60)
            print(f"  PROCESSANDO ANO: {ano}")
            print("=" * 60)


            cursor.execute(
                "SELECT cd_proposicoes FROM proposicoes WHERE ano = %s",
                (ano,)
            )
            ids_ano = {row[0] for row in cursor.fetchall()}

            if not ids_ano:
                print(f"  ⚠ Nenhuma proposição encontrada no banco para {ano}. Pulando...")
                continue

            print(f"  → {len(ids_ano)} proposições no banco para {ano}.")

            # =================================================================
            # ETAPA 2 — Atualizar codSituacao
            # =================================================================
            print(f"\n  ETAPA 2 — Baixando CSV de proposições de {ano}...")

            url_props = (
                f"http://dadosabertos.camara.leg.br/arquivos/proposicoes/"
                f"csv/proposicoes-{ano}.csv"
            )

            try:
                df_props = pd.read_csv(url_props, sep=';', low_memory=False)
                print(f"  ✓ CSV carregado: {len(df_props)} linhas.")

                if 'ultimoStatus_codSituacao' not in df_props.columns:
                    print(f"  ⚠ Coluna 'ultimoStatus_codSituacao' ausente em {ano}. Pulando etapa 2.")
                else:

                    df_filtrado = df_props[df_props['id'].isin(ids_ano)].copy()


                    df_filtrado.dropna(subset=['ultimoStatus_codSituacao'], inplace=True)


                    dados_situacao = list(zip(
                        df_filtrado['ultimoStatus_codSituacao'].astype(int),
                        df_filtrado['id'].astype(int)
                    ))

                    if not dados_situacao:
                        print(f"  ⚠ Nenhum codSituacao válido para atualizar em {ano}.")
                    else:
                        print(f"  → Atualizando {len(dados_situacao)} registros de codSituacao...")
                        cursor.executemany(
                            "UPDATE proposicoes SET codSituacao = %s WHERE cd_proposicoes = %s",
                            dados_situacao
                        )
                        conn.commit()
                        print(f"  ✓ codSituacao atualizado com sucesso para {ano}.")

            except Exception as e_etapa2:

                print(f"  ✗ ERRO na Etapa 2 (codSituacao) para {ano}: {e_etapa2}")
                print("     → Continuando para a Etapa 3...\n")

            # =================================================================
            # ETAPA 3 — Atualizar autoria
            # =================================================================
            print(f"\n  ETAPA 3 — Baixando CSV de autores de {ano}...")

            url_autores = (
                f"http://dadosabertos.camara.leg.br/arquivos/proposicoesAutores/"
                f"csv/proposicoesAutores-{ano}.csv"
            )

            try:
                df_autores = pd.read_csv(url_autores, sep=';', low_memory=False)
                print(f"  ✓ CSV de autores carregado: {len(df_autores)} linhas.")

                colunas_necessarias = ['idProposicao', 'idDeputadoAutor', 'proponente']
                colunas_ausentes = [c for c in colunas_necessarias if c not in df_autores.columns]

                if colunas_ausentes:
                    print(f"  ⚠ Colunas ausentes no CSV de autores de {ano}: {colunas_ausentes}. Pulando etapa 3.")
                else:

                    df_aut_filtrado = df_autores[
                        df_autores['idProposicao'].isin(ids_ano)
                    ].copy()

                    # --- HIGIENIZAÇÃO --
                    df_aut_filtrado['idDeputadoAutor'] = pd.to_numeric(
                        df_aut_filtrado['idDeputadoAutor'], errors='coerce'
                    )

                    df_aut_filtrado.dropna(subset=['idDeputadoAutor'], inplace=True)


                    df_aut_filtrado['proponente'] = pd.to_numeric(
                        df_aut_filtrado['proponente'], errors='coerce'
                    ).fillna(0).astype(int)


                    dados_autor = list(zip(
                        df_aut_filtrado['proponente'],
                        df_aut_filtrado['idProposicao'].astype(int),
                        df_aut_filtrado['idDeputadoAutor'].astype(int)
                    ))

                    if not dados_autor:
                        print(f"  ⚠ Nenhum dado de autoria válido para atualizar em {ano}.")
                    else:
                        total = len(dados_autor)
                        BATCH_SIZE = 500
                        atualizados = 0
                        inicio = time.time()

                        print(f"  → Iniciando atualização de {total} registros em lotes de {BATCH_SIZE}...")

                        for i in range(0, total, BATCH_SIZE):
                            lote = dados_autor[i : i + BATCH_SIZE]

                            cursor.executemany(
                                """
                                UPDATE proposicao_deputados
                                SET autor = %s
                                WHERE fk_proposicao = %s AND fk_deputado = %s
                                """,
                                lote
                            )
                            conn.commit()

                            atualizados += len(lote)
                            pct = (atualizados / total) * 100
                            elapsed = time.time() - inicio

                            eta = (elapsed / atualizados) * (total - atualizados) if atualizados else 0

                            print(
                                f"    [{atualizados:>6}/{total}] "
                                f"{pct:5.1f}% | "
                                f"decorrido: {elapsed:5.1f}s | "
                                f"restante estimado: {eta:5.1f}s"
                            )

                        print(f"  ✓ Autoria (proponente) atualizada com sucesso para {ano}. "
                              f"Total: {atualizados} registros em {time.time()-inicio:.1f}s.")

            except Exception as e_etapa3:
                print(f"  ✗ ERRO na Etapa 3 (autoria) para {ano}: {e_etapa3}")
                print("     → Continuando para o próximo ano...\n")

        print("\n" + "=" * 60)
        print("  Todos os anos foram processados.")
        print("=" * 60)

except Exception as e_geral:

    print(f"\n✗ ERRO CRÍTICO: {e_geral}")

finally:
    # =========================================================================
    # ETAPA 4 — Encerramento seguro da conexão
    # =========================================================================
    print("\n" + "=" * 60)
    print("ETAPA 4 — Encerrando conexão com o banco...")
    print("=" * 60)
    if conn is not None and conn.is_connected():
        cursor.close()
        conn.close()
        print("✓ Conexão encerrada com segurança.\n")
    else:
        print("⚠ Conexão não estava ativa. Nada a encerrar.\n")

In [ ]:
import mysql.connector
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import deque

DB_CONFIG = {
    "host": "54.198.148.230",
    "port": 3306,
    "user": "root",
    "password": "lumina1234",
    "database": "Lumina2"
}

API_BASE       = "https://dadosabertos.camara.leg.br/api/v2/proposicoes"
WORKERS        = 10
BATCH_COMMIT   = 500
MAX_TENTATIVAS = 3
PRINT_CADA     = 100

def buscar_id(id_prop: int):
    url = f"{API_BASE}/{id_prop}"
    for tentativa in range(1, MAX_TENTATIVAS + 1):
        try:
            resp = requests.get(url, timeout=15)
            if resp.status_code == 200:
                dados  = resp.json().get("dados", {})
                status = dados.get("statusProposicao") or {}
                cod    = status.get("codSituacao")


                return (id_prop, int(cod) if cod is not None else 0)

            elif resp.status_code == 404:
                return (id_prop, 0)

            elif resp.status_code == 429:
                time.sleep(15 * tentativa)
            else:
                time.sleep(3 * tentativa)

        except requests.exceptions.Timeout:
            time.sleep(5 * tentativa)
        except Exception:
            time.sleep(3 * tentativa)

    return (id_prop, 0)

print("=" * 65)
print("ETAPA 0 — Conectando ao banco de dados...")
print("=" * 65)

conn = None

try:
    conn = mysql.connector.connect(**DB_CONFIG)
    cursor = conn.cursor()
    print("✓ Conexão estabelecida.\n")

    print("=" * 65)
    print("ETAPA 1 — Buscando proposições com codSituacao NULL...")
    print("=" * 65)

    cursor.execute("""
        SELECT cd_proposicoes
        FROM proposicoes
        WHERE codSituacao IS NULL
        ORDER BY cd_proposicoes
    """)
    ids_pendentes = [row[0] for row in cursor.fetchall()]
    total         = len(ids_pendentes)

    if total == 0:
        print("✓ Nenhuma proposição pendente. Tudo já está atualizado!")
    else:
        print(f"✓ {total} proposições com codSituacao NULL encontradas.")
        print(f"✓ {WORKERS} threads paralelas.")
        est_min = (total * 0.5) / WORKERS / 60
        print(f"  ⏱ Estimativa: ~{est_min:.0f} min\n")

        print("=" * 65)
        print("ETAPA 2 — Consultando API em paralelo e atualizando banco...")
        print(f"  → Progresso exibido a cada {PRINT_CADA} IDs concluídos.")
        print(f"  → Commit no banco a cada {BATCH_COMMIT} registros salvos.")
        print(f"  → IDs sem codSituacao na API recebem 0 (para não reprocessar).")
        print("=" * 65)

        atualizados    = 0
        com_cod_real   = 0
        marcados_zero  = 0
        concluidos     = 0
        lote_dados     = []
        inicio         = time.time()
        tempos         = deque(maxlen=200)

        with ThreadPoolExecutor(max_workers=WORKERS) as executor:
            futuros = {
                executor.submit(buscar_id, id_prop): id_prop
                for id_prop in ids_pendentes
            }

            for futuro in as_completed(futuros):
                t0 = time.time()
                try:
                    id_prop, cod = futuro.result()
                except Exception:
                    concluidos += 1
                    continue

                lote_dados.append((cod, id_prop))

                if cod > 0:
                    com_cod_real += 1
                else:
                    marcados_zero += 1

                concluidos += 1
                tempos.append(time.time() - t0)

                if len(lote_dados) >= BATCH_COMMIT:
                    cursor.executemany(
                        "UPDATE proposicoes SET codSituacao = %s WHERE cd_proposicoes = %s",
                        lote_dados
                    )
                    conn.commit()
                    atualizados += len(lote_dados)
                    lote_dados   = []
                    print(f"  ✓ COMMIT — {atualizados} registros salvos no banco até agora.")

                if concluidos % PRINT_CADA == 0 or concluidos == total:
                    elapsed   = time.time() - inicio
                    restantes = total - concluidos
                    eta_s     = (restantes / WORKERS) * (sum(tempos) / len(tempos)) if tempos else 0
                    print(
                        f"  [{concluidos:>6}/{total}] "
                        f"{concluidos/total*100:5.1f}% | "
                        f"com código: {com_cod_real:>6} | "
                        f"sem código (0): {marcados_zero:>6} | "
                        f"decorrido: {elapsed/60:.1f}min | "
                        f"rest.: ~{eta_s/60:.1f}min"
                    )

        if lote_dados:
            cursor.executemany(
                "UPDATE proposicoes SET codSituacao = %s WHERE cd_proposicoes = %s",
                lote_dados
            )
            conn.commit()
            atualizados += len(lote_dados)

        elapsed_total = time.time() - inicio
        print("\n" + "=" * 65)
        print("  RESUMO FINAL")
        print("=" * 65)
        print(f"  ✓ Total salvo no banco         : {atualizados}")
        print(f"  ✓ Com codSituacao real          : {com_cod_real}")
        print(f"  ⚠ Sem codSituacao (gravado 0)  : {marcados_zero}")
        print(f"  ⏱ Tempo total                  : {elapsed_total/60:.1f} min")

except Exception as e_geral:
    print(f"\n✗ ERRO CRÍTICO: {e_geral}")

finally:
    print("\n" + "=" * 65)
    print("ETAPA 3 — Encerrando conexão...")
    print("=" * 65)
    if conn is not None and conn.is_connected():
        cursor.close()
        conn.close()
        print("✓ Conexão encerrada com segurança.\n")
    else:
        print("⚠ Conexão não estava ativa.\n")

## Tabela proposicao_deputados

### coluna tema e peso_tema

In [ ]:
cursor = conn.cursor()

sql_tema = """
UPDATE proposicao_deputados pd
JOIN tema_proposicoes tp ON CAST(tp.id_proposicao AS UNSIGNED) = CAST(pd.fk_proposicao AS UNSIGNED)
SET pd.tema = tp.id_tema;
"""

try:
    cursor.execute(sql_tema)

    # O rowcount diz quantas linhas foram REALMENTE alteradas
    print(f"Linhas encontradas e modificadas: {cursor.rowcount}")

    if cursor.rowcount > 0:
        conn.commit()
        print("✅ Dados salvos com sucesso!")
    else:
        print("⚠️ Nenhuma linha foi alterada. Os IDs de proposição podem não ser iguais nas duas tabelas.")

except Exception as e:
    print(f"❌ Erro: {e}")

cursor.close()

In [ ]:
cursor = conn.cursor()

# Comando para mudar o tipo de VARCHAR para INT
sql_alterar = "ALTER TABLE proposicao_deputados MODIFY COLUMN tema INT;"

try:
    cursor.execute(sql_alterar)
    conn.commit()
    print("✅ Coluna 'tema' alterada para INT com sucesso!")
except Exception as e:
    print(f"❌ Erro ao alterar coluna: {e}")
    print("Dica: Verifique se existem textos na coluna que não podem virar números.")

cursor.close()

In [ ]:
def inspecionar_estrutura(tabela):
    try:
        cursor.execute(f"DESCRIBE Lumina2.{tabela}")
        estrutura = cursor.fetchall()
        print(f"\n📋 ESTRUTURA DA TABELA: {tabela}")
        print("-" * 60)
        for coluna in estrutura:
            print(f" Campo: {coluna[0]:<20} | Tipo: {coluna[1]:<15} | Nulo: {coluna[2]}")
    except Exception as e:
        print(f"❌ Erro ao ler a tabela {tabela}: {e}")

# Execução do diagnóstico preciso
inspecionar_estrutura("proposicao_deputados")
inspecionar_estrutura("proposicoes")
inspecionar_estrutura("tema_proposicoes")
inspecionar_estrutura("tipo_peso")
inspecionar_estrutura("tema_peso")

In [ ]:
try:
    # Captura a listagem real de tabelas existentes no schema Lumina2
    cursor.execute("SHOW TABLES FROM Lumina2")
    tabelas = [linha[0] for linha in cursor.fetchall()]
    print("📋 Tabelas encontradas fisicamente no banco Lumina2:")
    print("-" * 50)
    for t in tabelas:
        print(f" - {t}")
except Exception as e:
    print(f"❌ Erro ao listar tabelas: {e}")